In [9]:

#----------------------------
# COMBINAR CSV NDSI y NDVI en uno solo
#-----------------------------

import os
import pandas as pd

# === Rutas base ===
base_ndsi = r"M:\material_isa\estadisticos\_Turberas\indices_analisis_NDSI"
base_ndvi = r"M:\material_isa\estadisticos\_Turberas\indices_analisis_NDVI"
output_dir = r"M:\material_isa\estadisticos\_Turberas\NDSI_vs_NDVI"

# Asegura que la carpeta de salida exista
os.makedirs(output_dir, exist_ok=True)

# Obtener lista de turberas desde archivos NDSI
turberas = [f.split("_analisis_NDSI_v2.csv")[0] for f in os.listdir(base_ndsi) if f.endswith("_analisis_NDSI_v2.csv")]

for turbera in turberas:
    path_ndsi = os.path.join(base_ndsi, f"{turbera}_analisis_NDSI_v2.csv")
    path_ndvi = os.path.join(base_ndvi, f"{turbera}_analisis_NDVI.csv")
    path_output = os.path.join(output_dir, f"{turbera}_NDSI_vs_NDVI_v2.csv")

    # Verifica existencia de ambos archivos
    if not os.path.exists(path_ndsi) or not os.path.exists(path_ndvi):
        print(f"[⚠️] Archivos faltantes para {turbera}, omitiendo...")
        continue

    try:
        # Leer archivos
        df_ndsi = pd.read_csv(path_ndsi, sep=";")
        df_ndvi = pd.read_csv(path_ndvi, sep=";")

        # Crear columna YEAR desde "Temporada" (ej. 2015-2016 -> 2016)
        df_ndsi["YEAR"] = df_ndsi["Temporada"].apply(lambda x: int(x.split("-")[1]))

        # Merge por YEAR
        df_merged = pd.merge(df_ndsi, df_ndvi, on="YEAR", how="inner")

        # Guardar CSV combinado
        df_merged.to_csv(path_output, index=False, sep=";")
        print(f"[✅] Combinado guardado: {turbera}")

    except Exception as e:
        print(f"[❌] Error procesando {turbera}: {e}")

#----------------------------
# GRAFICAR dispersion todas los parametros de NDSI vs todos los de NDVI
#-----------------------------

import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ------------------------
# CONFIGURACIÓN INICIAL
# ------------------------

# Ruta a los CSV combinados NDSI vs NDVI
input_dir = r"M:\material_isa\estadisticos\_Turberas\NDSI_vs_NDVI"

# Variables a graficar: pares (x, y) y sus títulos
scatter_pairs = [
    ("Periodo (dias)", "NDVI Verdor Max", "Duración de Nieve", "NDVI Máximo"),
    ("Periodo (dias)", "MEDIA NDVI", "Duración de Nieve", "NDVI Medio"),
    ("Persistencia (%)", "NDVI Verdor Max", "Persistencia", "NDVI Máximo"),
    ("Persistencia (%)", "MEDIA NDVI", "Persistencia", "NDVI Medio"),
    ("Periodo (dias)", "Día Juliano", "Duración de Nieve", "Fecha NDVI Máximo (día juliano)"),
    ("Persistencia (%)", "Día Juliano", "Persistencia", "Fecha NDVI Máximo (día juliano)")
]

# Colores de línea por turbera (puedes expandir según tus archivos)
colores_turberas = {
    "Bassa Nera": "#1f77b4",
    "Clots de Rialba": "#ff7f0e",
    "Estanyeres": "#2ca02c",
    "Pla de Sartari": "#9467bd",
    "Plan deth Horn": "#d62728",
    "Pleta de Molieres": "#8c564b"
}

# ------------------------
# LECTURA Y COMBINACIÓN
# ------------------------

# Leer solo los archivos relevantes que terminan en _NDSI_vs_NDVI_v2.csv
dfs = []
for file in os.listdir(input_dir):
    if file.endswith("_NDSI_vs_NDVI_v2.csv"):
        turbera = file.replace("_NDSI_vs_NDVI_v2.csv", "").replace("_", " ")
        df = pd.read_csv(os.path.join(input_dir, file), sep=";")
        df["Turbera"] = turbera
        dfs.append(df)


# Unir todos en un solo dataframe
df_total = pd.concat(dfs, ignore_index=True)

# Calcular Día Juliano del Verdor Máx si no está
if "Verdor Max" in df_total.columns and "Día Juliano" not in df_total.columns:
    df_total["Verdor Max"] = pd.to_datetime(df_total["Verdor Max"], errors="coerce")
    df_total["Día Juliano"] = df_total["Verdor Max"].dt.dayofyear

# ------------------------
# GRÁFICOS DE DISPERSIÓN
# ------------------------

# Crear carpeta de salida
output_dir = os.path.join(input_dir, "graficos_dispersion")
os.makedirs(output_dir, exist_ok=True)

for i, (x, y, x_label, y_label) in enumerate(scatter_pairs, start=1):
    plt.figure(figsize=(8, 6))

    # Gráfico con leyenda por turbera
    for turbera, grupo in df_total.groupby("Turbera"):
        plt.scatter(grupo[x], grupo[y], label=turbera, alpha=0.7,
                    s=60, edgecolor='black', color=colores_turberas.get(turbera, None))

    plt.xlabel(x_label)
    plt.ylabel(y_label)
    plt.title(f"{y_label} vs {x_label}")
    plt.legend(title="Turbera", loc="best")
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()

    # Guardar gráfico
    nombre_archivo = f"scatter_{y.replace(' ', '_')}_vs_{x.replace(' ', '_')}_v2.png"
    plt.savefig(os.path.join(output_dir, nombre_archivo), dpi=150)
    plt.close()

print("✔ Todos los gráficos de dispersión se han generado correctamente.")




[✅] Combinado guardado: Bassa_Nera
[✅] Combinado guardado: Clots_de_Rialba
[✅] Combinado guardado: Estanyeres
[✅] Combinado guardado: Plan_deth_Horn
[✅] Combinado guardado: Pla_de_Sartari
[✅] Combinado guardado: Pleta_de_Molieres
✔ Todos los gráficos de dispersión se han generado correctamente.


In [13]:

# --------------------------
# GRÁFICOS doble eje (con día juliano) - Versión v2
# --------------------------

import pandas as pd
import matplotlib.pyplot as plt
import os

# ------------------------
# RUTAS
# ------------------------
input_dir = r"M:\material_isa\estadisticos\_Turberas\NDSI_vs_NDVI"
output_dir = os.path.join(input_dir, "sub_plots_v2")
os.makedirs(output_dir, exist_ok=True)

# ------------------------
# LÍMITES POR VARIABLE
# ------------------------
limits = {
    'Periodo (dias)': (25, 210),
    'Persistencia (%)': (20, 105),
    'NDVI Verdor Max': (0.2, 0.8),
    'MEDIA NDVI': (0.1, 0.6),
    'DIA_VERDOR_MAX': (140, 260)
}

# ------------------------
# ETIQUETAS DE LOS EJES
# ------------------------
labels = {
    'Periodo (dias)': 'Duración de Nieve (días)',
    'Persistencia (%)': 'Persistencia Nival (%)',
    'NDVI Verdor Max': 'NDVI Máximo',
    'MEDIA NDVI': 'NDVI Medio',
    'DIA_VERDOR_MAX': 'Fecha NDVI Máx (día año)'
}

# ------------------------
# TURBERAS DISPONIBLES
# ------------------------
turberas = [
    'Bassa_Nera', 'Plan_deth_Horn', 'Pla_de_Sartari',
    'Pleta_de_Molieres', 'Clots_de_Rialba', 'Estanyeres'
]

# ------------------------
# COMBINACIONES A GRAFICAR
# ------------------------
combinations = [
    ('Periodo (dias)', 'NDVI Verdor Max'),
    ('Periodo (dias)', 'MEDIA NDVI'),
    ('Periodo (dias)', 'DIA_VERDOR_MAX'),
    ('Persistencia (%)', 'NDVI Verdor Max'),
    ('Persistencia (%)', 'MEDIA NDVI'),
    ('Persistencia (%)', 'DIA_VERDOR_MAX'),
]

# ------------------------
# FUNCIÓN DE GRAFICADO
# ------------------------
def generar_subplots(y1, y2):
    fig, axes = plt.subplots(3, 2, figsize=(14, 10), sharex=True)
    axes = axes.flatten()

    for i, turbera in enumerate(turberas):
        filepath = os.path.join(input_dir, f"{turbera}_NDSI_vs_NDVI_v2.csv")
        if not os.path.exists(filepath):
            continue

        df = pd.read_csv(filepath, sep=';')

        # Calcular día del año si es necesario
        if 'DIA_VERDOR_MAX' not in df.columns and 'Verdor Max' in df.columns:
            df['DIA_VERDOR_MAX'] = pd.to_datetime(df['Verdor Max'], errors='coerce').dt.dayofyear

        df = df.dropna(subset=[y1, y2])

        ax1 = axes[i]
        ax2 = ax1.twinx()

        ax1.plot(df['YEAR'], df[y1], 'o-', color='dodgerblue')
        ax2.plot(df['YEAR'], df[y2], 's-', color='crimson')

        ax1.set_ylim(limits[y1])
        ax2.set_ylim(limits[y2])

        ax1.set_title(turbera.replace("_", " "))
        ax1.set_xlabel('Año')
        ax1.set_ylabel(labels[y1], color='dodgerblue')
        ax2.set_ylabel(labels[y2], color='crimson')

        ax1.tick_params(axis='y', labelcolor='dodgerblue')
        ax2.tick_params(axis='y', labelcolor='crimson')

        ax1.set_xticks(df['YEAR'])
        ax1.grid(True, which='both', axis='x', linestyle=':', color='lightgray')

    plt.suptitle(f"{labels[y1]} vs {labels[y2]}", fontsize=14)

    if 'Persistencia (%)' in [y1, y2]:
        leyenda = (
            "6 = Muy persistente (0–2 caídas NDSI<0.4) | 5 = Persistente (3–4 veces) | "
            "4 = Medianamente persistente (5–6 veces)\n"
            "3 = Poco persistente (7–8 veces) | 2 = Muy poco persistente (9–10 veces) | "
            "1 = Casi nula persistencia (11–12 veces)"
        )
        #plt.figtext(0.5, 0.007, leyenda, wrap=True, horizontalalignment='center', fontsize=9)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])

    filename = f"{y1}_vs_{y2}.png".lower().replace(" ", "_").replace("día_año", "dia_anyo").replace("(", "").replace(")", "")
    filepath = os.path.join(output_dir, filename)
    plt.savefig(filepath, dpi=300)
    plt.close()

# ------------------------
# EJECUTAR
# ------------------------
for y1, y2 in combinations:
    generar_subplots(y1, y2)

print("✔ Todos los gráficos de sub-plot v2 se han generado correctamente.")


✔ Todos los gráficos de sub-plot v2 se han generado correctamente.


In [15]:
import pandas as pd
import os
from scipy.stats import pearsonr, spearmanr

# -----------------------------
# RUTA DE ENTRADA Y SALIDA
# -----------------------------
input_dir = r"M:\material_isa\estadisticos\_Turberas\NDSI_vs_NDVI"
output_path = os.path.join(input_dir, "correlaciones_por_turbera_v2.csv")

# -----------------------------
# VARIABLES A ANALIZAR
# -----------------------------
combinations = [
    ('Periodo (dias)', 'NDVI Verdor Max'),
    ('Periodo (dias)', 'MEDIA NDVI'),
    ('Periodo (dias)', 'DIA_VERDOR_MAX'),
    ('Persistencia (%)', 'NDVI Verdor Max'),
    ('Persistencia (%)', 'MEDIA NDVI'),
    ('Persistencia (%)', 'DIA_VERDOR_MAX'),
]

# -----------------------------
# CALCULAR CORRELACIONES
# -----------------------------
resultados = []

for file in os.listdir(input_dir):
    if not file.endswith("_NDSI_vs_NDVI_v2.csv"):
        continue

    turbera = file.replace("_NDSI_vs_NDVI_v2.csv", "").replace("_", " ")
    df = pd.read_csv(os.path.join(input_dir, file), sep=';')

    # Calcular día juliano si no está
    if "DIA_VERDOR_MAX" not in df.columns and "Verdor Max" in df.columns:
        df["Verdor Max"] = pd.to_datetime(df["Verdor Max"], errors='coerce')
        df["DIA_VERDOR_MAX"] = df["Verdor Max"].dt.dayofyear

    for x, y in combinations:
        df_corr = df[[x, y]].dropna()

        if len(df_corr) >= 3:
            pearson, _ = pearsonr(df_corr[x], df_corr[y])
            spearman, _ = spearmanr(df_corr[x], df_corr[y])
        else:
            pearson = spearman = None

        resultados.append({
            "Turbera": turbera,
            "X": x,
            "Y": y,
            "Pearson": round(pearson, 3) if pearson is not None else None,
            "Spearman": round(spearman, 3) if spearman is not None else None
        })

# -----------------------------
# GUARDAR RESULTADOS
# -----------------------------
df_resultados = pd.DataFrame(resultados)
df_resultados.to_csv(output_path, index=False, sep=";")
print(f"✅ Correlaciones guardadas en: {output_path}")



✅ Correlaciones guardadas en: M:\material_isa\estadisticos\_Turberas\NDSI_vs_NDVI\correlaciones_por_turbera_v2.csv


In [17]:
import pandas as pd
import os

# -----------------------------------
# RUTA DE ENTRADA Y SALIDA
# -----------------------------------
input_path = r"M:\material_isa\estadisticos\_Turberas\NDSI_vs_NDVI\correlaciones_por_turbera_v2.csv"
output_path = input_path.replace(".csv", "_interpretado.csv")

# -----------------------------------
# CARGAR DATOS
# -----------------------------------
df = pd.read_csv(input_path, sep=";")

# -----------------------------------
# INTERPRETACIÓN DE CORRELACIÓN
# -----------------------------------
def interpretar_corr(valor):
    if pd.isna(valor):
        return "Sin datos"
    elif abs(valor) >= 0.7:
        return "Correlación fuerte"
    elif abs(valor) >= 0.5:
        return "Correlación moderada"
    elif abs(valor) >= 0.3:
        return "Correlación débil"
    else:
        return "Casi nula o muy débil"

# Aplicar interpretación
df["Interpretación Pearson"] = df["Pearson"].apply(interpretar_corr)
df["Interpretación Spearman"] = df["Spearman"].apply(interpretar_corr)

# -----------------------------------
# GUARDAR RESULTADOS
# -----------------------------------
df.to_csv(output_path, sep=";", index=False)
print(f"✅ Archivo interpretado guardado en:\n{output_path}")



✅ Archivo interpretado guardado en:
M:\material_isa\estadisticos\_Turberas\NDSI_vs_NDVI\correlaciones_por_turbera_v2_interpretado.csv


In [23]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os

# Leer datos
df = pd.read_csv(r"M:\material_isa\estadisticos\_Turberas\NDSI_vs_NDVI\correlaciones_por_turbera_v2_interpretado.csv", sep=";")

# Diccionario de renombres personalizados
renombrar_x = {
    "Periodo (dias)": "Duración",
    "Persistencia (%)": "Persistencia"
}

renombrar_y = {
    "MEDIA NDVI": "Media anual\nde NDVI",
    "NDVI Verdor Max": "NDVI Máximo\nanual",
    "DIA_VERDOR_MAX": "Fecha del\nNDVI Max"
}

# Orden personalizado (coincide con columnas del CSV)
orden_x = ["Periodo (dias)", "Persistencia (%)"]
orden_y = ["MEDIA NDVI", "NDVI Verdor Max", "DIA_VERDOR_MAX"]

# Directorio de salida
output_dir = r"M:\material_isa\estadisticos\_Turberas\NDSI_vs_NDVI\correlacion\pearson"
os.makedirs(output_dir, exist_ok=True)

# Lista de turberas
turberas = df["Turbera"].unique()

for turbera in turberas:
    df_t = df[df["Turbera"] == turbera]

    # Crear matriz con orden específico
    matriz = df_t.pivot(index="X", columns="Y", values="Pearson").loc[orden_x, orden_y]

    # Renombrar etiquetas
    matriz.index = [renombrar_x[i] for i in matriz.index]
    matriz.columns = [renombrar_y[j] for j in matriz.columns]

    # Crear el heatmap
    plt.figure(figsize=(8, 4))
    sns.heatmap(matriz, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, cbar_kws={"label": "Coef. de Pearson"})

    # Etiquetas de ejes en negrita
    plt.xlabel("Vegetación (Y)", fontsize=12)
    plt.ylabel("Nieve (X)", fontsize=12)
    plt.yticks(fontsize=10)
    plt.yticks(rotation=90, va='center', ha='center')
    ax = plt.gca()
    for label in ax.get_yticklabels():
        label.set_rotation(90)
        label.set_ha('center')
        label.set_va('center')

    # Título
    plt.title(f"Matriz de Correlación (Pearson) – {turbera}", fontsize=14)

    # Guardar
    output_path = os.path.join(output_dir, f"heatmap_pearson_{turbera.lower().replace(' ', '_')}_v2.png")
    plt.tight_layout()
    plt.savefig(output_path, dpi=300)
    plt.close()

print("✅ Heatmaps actualizados con nombres personalizados.")


✅ Heatmaps actualizados con nombres personalizados.


In [25]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os

# Leer datos corregidos
df = pd.read_csv(r"M:\material_isa\estadisticos\_Turberas\NDSI_vs_NDVI\correlaciones_por_turbera_v2_interpretado.csv", sep=";")

# Diccionario de renombres personalizados
renombrar_x = {
    "Periodo (dias)": "Duración",
    "Persistencia (%)": "Persistencia"
}

renombrar_y = {
    "MEDIA NDVI": "Media anual\nde NDVI",
    "NDVI Verdor Max": "NDVI Máximo\nanual",
    "DIA_VERDOR_MAX": "Fecha del\nNDVI Max"
}

# Orden personalizado
orden_x = ["Periodo (dias)", "Persistencia (%)"]
orden_y = ["MEDIA NDVI", "NDVI Verdor Max", "DIA_VERDOR_MAX"]

# Directorio de salida para Spearman
output_dir = r"M:\material_isa\estadisticos\_Turberas\NDSI_vs_NDVI\correlacion\spearman"
os.makedirs(output_dir, exist_ok=True)

# Lista de turberas
turberas = df["Turbera"].unique()

for turbera in turberas:
    df_t = df[df["Turbera"] == turbera]

    # Crear matriz con Spearman y orden fijo
    matriz = df_t.pivot(index="X", columns="Y", values="Spearman").loc[orden_x, orden_y]

    # Renombrar etiquetas
    matriz.index = [renombrar_x[i] for i in matriz.index]
    matriz.columns = [renombrar_y[j] for j in matriz.columns]

    # Crear el heatmap
    plt.figure(figsize=(8, 4))
    sns.heatmap(matriz, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1,
                cbar_kws={"label": "Coef. de Spearman"})

    # Etiquetas de ejes en negrita
    plt.xlabel("Vegetación (Y)", fontsize=12)
    plt.ylabel("Nieve (X)", fontsize=12)
    plt.yticks(fontsize=10, rotation=90, va='center', ha='center')
    ax = plt.gca()
    for label in ax.get_yticklabels():
        label.set_rotation(90)
        label.set_ha('center')
        label.set_va('center')

    # Título
    plt.title(f"Matriz de Correlación (Spearman) – {turbera}", fontsize=14)

    # Guardar
    output_path = os.path.join(output_dir, f"heatmap_spearman_{turbera.lower().replace(' ', '_')}_v2.png")
    plt.tight_layout()
    plt.savefig(output_path, dpi=300)
    plt.close()

print("✅ Heatmaps de Spearman actualizados correctamente.")


✅ Heatmaps de Spearman actualizados correctamente.
